# Fine-Tuned Model Evaluation

This notebook evaluates all fine-tuned LoRA adapters under `fine_tuned_models/` and compares them with the GPT-4o rewrite already stored in the uploaded dataset's `reversed_headline` field.

Upload either `nonsarcastic_to_sarcastic_out_classifier_correct_similarity_lt_0p4.json` or `sarcastic_to_nonsarcastic_out_classifier_correct_similarity_lt_0p4.json` at runtime. The notebook uses `headline` as model input, `is_sarcastic` as the original headline label, and ignores `reversed_headline` during fine-tuned model inference. `reversed_headline` is used only as the GPT-4o benchmark output.

Metrics match `baseline_evaluation.ipynb`: sarcasm-flip classifier accuracy, cosine similarity, and perplexity.

Because the fine-tuned model folders are too large for GitHub, the Colab setup asks you to upload archives for the local `flan_t5` and `gpt-2` folders right after cloning the code. Zip each folder before uploading, for example `flan_t5.zip` and `gpt-2.zip`; the notebook extracts them back into `fine_tuned_models/flan_t5` and `fine_tuned_models/gpt-2`.

In [ ]:
# Pull code, install dependencies

import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/LINCHENYU2030S/CS4248_group_project.git"
REPO_ROOT = Path("/content/CS4248_group_project")

current_dir = Path.cwd().resolve()
if current_dir.name == "CS4248_group_project":
    project_root = current_dir
elif current_dir.name in {"notebooks", "evaluation_methods"} and current_dir.parent.name == "CS4248_group_project":
    project_root = current_dir.parent
elif (current_dir / "evaluation_methods").exists():
    project_root = current_dir
elif REPO_ROOT.exists():
    project_root = REPO_ROOT
elif "google.colab" in sys.modules:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    project_root = REPO_ROOT
else:
    raise RuntimeError(f"Could not locate project root from: {current_dir}")

os.chdir(project_root)
PROJECT_ROOT = Path.cwd().resolve()
EVALUATION_METHODS_DIR = PROJECT_ROOT / "evaluation_methods"
FINE_TUNED_MODELS_DIR = PROJECT_ROOT / "fine_tuned_models"

if not EVALUATION_METHODS_DIR.exists():
    raise RuntimeError(f"Missing evaluation methods directory: {EVALUATION_METHODS_DIR}")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
for path in [PROJECT_ROOT, EVALUATION_METHODS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "transformers>=4.39.0",
        "peft>=0.10.0",
        "sentence-transformers>=2.6.0",
        "accelerate>=0.28.0",
        "safetensors>=0.4.0",
        "seaborn>=0.13.0",
        "ipywidgets>=8.1.0",
        "sentencepiece>=0.1.99",
    ],
    check=True,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Evaluation methods: {EVALUATION_METHODS_DIR}")
print(f"Fine-tuned models: {FINE_TUNED_MODELS_DIR}")


In [ ]:
# Upload fine-tuned model folders at run time

import os
import shutil
import tarfile
import zipfile

REQUIRED_MODEL_FOLDERS = ("flan_t5", "gpt-2")
UPLOAD_MODEL_ARCHIVES = True  # In Colab, upload flan_t5.zip and gpt-2.zip in this cell.

FINE_TUNED_MODELS_DIR.mkdir(parents=True, exist_ok=True)


def _is_safe_extract_path(base_dir: Path, target_path: Path) -> bool:
    base_dir = base_dir.resolve()
    target_path = target_path.resolve()
    return target_path == base_dir or str(target_path).startswith(str(base_dir) + os.sep)


def _safe_extract_zip(archive_path: Path, extract_dir: Path) -> None:
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target_path = extract_dir / member.filename
            if not _is_safe_extract_path(extract_dir, target_path):
                raise ValueError(f"Unsafe path in zip archive {archive_path.name}: {member.filename}")
        archive.extractall(extract_dir)


def _safe_extract_tar(archive_path: Path, extract_dir: Path) -> None:
    with tarfile.open(archive_path) as archive:
        for member in archive.getmembers():
            target_path = extract_dir / member.name
            if not _is_safe_extract_path(extract_dir, target_path):
                raise ValueError(f"Unsafe path in tar archive {archive_path.name}: {member.name}")
        archive.extractall(extract_dir)


def _copy_or_replace(source_dir: Path, target_dir: Path) -> None:
    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_dir, target_dir)


def _direct_folder_source(unpack_dir: Path) -> Path:
    children = [path for path in unpack_dir.iterdir() if path.name != "__MACOSX"]
    if len(children) == 1 and children[0].is_dir():
        return children[0]
    return unpack_dir


def _install_uploaded_model_archive(upload_name: str, payload: bytes) -> set[str]:
    archive_path = PROJECT_ROOT / upload_name
    archive_path.write_bytes(payload)

    unpack_dir = PROJECT_ROOT / "_uploaded_fine_tuned_models" / upload_name.replace("/", "_").replace(".", "_")
    if unpack_dir.exists():
        shutil.rmtree(unpack_dir)
    unpack_dir.mkdir(parents=True, exist_ok=True)

    lower_name = upload_name.lower()
    if lower_name.endswith(".zip"):
        _safe_extract_zip(archive_path, unpack_dir)
    elif lower_name.endswith((".tar", ".tar.gz", ".tgz")):
        _safe_extract_tar(archive_path, unpack_dir)
    else:
        raise ValueError(f"Unsupported model archive type: {upload_name}. Upload .zip, .tar, .tar.gz, or .tgz files.")

    installed_folders = set()
    for folder_name in REQUIRED_MODEL_FOLDERS:
        candidates = [path for path in unpack_dir.rglob(folder_name) if path.is_dir()]
        if candidates:
            _copy_or_replace(candidates[0], FINE_TUNED_MODELS_DIR / folder_name)
            installed_folders.add(folder_name)

    if installed_folders:
        return installed_folders

    if "flan" in lower_name:
        folder_name = "flan_t5"
    elif "gpt" in lower_name:
        folder_name = "gpt-2"
    else:
        raise ValueError(
            f"Could not infer target model folder from {upload_name}. "
            "Name the archive flan_t5.zip or gpt-2.zip, or include that folder inside the archive."
        )

    _copy_or_replace(_direct_folder_source(unpack_dir), FINE_TUNED_MODELS_DIR / folder_name)
    return {folder_name}


missing_model_folders = [folder for folder in REQUIRED_MODEL_FOLDERS if not (FINE_TUNED_MODELS_DIR / folder).exists()]
if "google.colab" in sys.modules and (UPLOAD_MODEL_ARCHIVES or missing_model_folders):
    from google.colab import files

    print("Upload the fine-tuned model folder archives now.")
    print("Expected archives: flan_t5.zip and gpt-2.zip, or tar/tar.gz/tgz equivalents.")
    print(f"They will be extracted under: {FINE_TUNED_MODELS_DIR}")
    uploaded_model_archives = files.upload()
    if not uploaded_model_archives:
        raise FileNotFoundError("No fine-tuned model archives were uploaded.")

    installed_model_folders = set()
    for upload_name, payload in uploaded_model_archives.items():
        installed_model_folders.update(_install_uploaded_model_archive(upload_name, payload))
    print("Installed model folders:", sorted(installed_model_folders))
elif missing_model_folders:
    raise FileNotFoundError(
        f"Missing local fine-tuned model folders: {missing_model_folders}. "
        f"Expected them under {FINE_TUNED_MODELS_DIR}."
    )

adapter_paths = sorted(FINE_TUNED_MODELS_DIR.glob("**/adapter_config.json"))
if not adapter_paths:
    raise FileNotFoundError(f"No PEFT adapter_config.json files found under {FINE_TUNED_MODELS_DIR}")

print("Found adapter folders:")
for adapter_config_path in adapter_paths:
    print("-", adapter_config_path.parent.relative_to(PROJECT_ROOT))


In [ ]:
# Upload evaluation dataset at run time

if "google.colab" not in sys.modules:
    raise RuntimeError("This upload cell is intended for Colab. In a local notebook, set DATASET_PATH manually.")

from google.colab import files

ALLOWED_DATASET_FILENAMES = {
    "nonsarcastic_to_sarcastic_out_classifier_correct_similarity_lt_0p4.json",
    "sarcastic_to_nonsarcastic_out_classifier_correct_similarity_lt_0p4.json",
}

print("Upload exactly one of these files:")
print(sorted(ALLOWED_DATASET_FILENAMES))
uploaded = files.upload()

uploaded_candidates = [name for name in uploaded if name in ALLOWED_DATASET_FILENAMES]
if len(uploaded_candidates) != 1:
    raise FileNotFoundError(
        "Expected exactly one supported uploaded JSON file. "
        f"Uploaded files: {list(uploaded)}"
    )

DATASET_FILENAME = uploaded_candidates[0]
DATASET_PATH = PROJECT_ROOT / DATASET_FILENAME
DATASET_PATH.write_bytes(uploaded[DATASET_FILENAME])

print(f"Dataset saved to: {DATASET_PATH}")


In [ ]:
# Imports and configuration

import gc
import json
import math
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from matplotlib import pyplot as plt
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer, set_seed

from evaluation_methods.text_similarity import batch_cosine_similarity
from evaluation_methods.text_perplexity import batch_perplexity
from evaluation_methods.utils import (
    build_finetuned_prompt,
    clean_generation,
    get_source_style,
    get_target_publication,
    get_target_style,
    load_sarcasm_classifier,
    predict_sarcasm_labels,
    sample_dataset,
)

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_colwidth", 120)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CLASSIFIER_DEVICE = "cpu" if DEVICE == "cuda" else DEVICE

DATASET_SAMPLE_PERCENT = 100.0  # Set below 100 for a faster evaluation subset.
CLASSIFIER_MODEL_NAME = "helinivan/english-sarcasm-detector"
CLASSIFIER_BATCH_SIZE = 64
CLASSIFIER_MAX_LENGTH = 256
PERPLEXITY_BATCH_SIZE = 8
SIMILARITY_BATCH_SIZE = 256
USE_FP16_ON_GPU = True

if not 0 < DATASET_SAMPLE_PERCENT <= 100:
    raise ValueError("DATASET_SAMPLE_PERCENT must be in the interval (0, 100].")

OUTPUT_STEM = Path(DATASET_FILENAME).stem
OUTPUT_DIR = PROJECT_ROOT / "fine_tuned_evaluation_outputs" / OUTPUT_STEM
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using device for generation/perplexity: {DEVICE}")
print(f"Using classifier device: {CLASSIFIER_DEVICE}")
print(f"Dataset sample percent: {DATASET_SAMPLE_PERCENT}%")
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
# Fine-tuned adapter specs

@dataclass(frozen=True)
class FineTunedModelSpec:
    key: str
    label: str
    adapter_path: Path
    base_model_name: str
    architecture: str
    comparison_group: str
    batch_size: int
    max_source_length: int = 256
    max_new_tokens: int = 64


MODEL_SPECS = [
    FineTunedModelSpec(
        key="gpt2_finetuned",
        label="GPT-2 fine-tuned",
        adapter_path=FINE_TUNED_MODELS_DIR / "gpt-2" / "gpt2-adapters",
        base_model_name="gpt2-large",
        architecture="causal",
        comparison_group="rank",
        batch_size=4,
    ),
    FineTunedModelSpec(
        key="flan_t5_r32",
        label="FLAN-T5 r32",
        adapter_path=FINE_TUNED_MODELS_DIR / "flan_t5" / "flan-t5_adapters_r32",
        base_model_name="google/flan-t5-base",
        architecture="seq2seq",
        comparison_group="rank",
        batch_size=8,
    ),
    FineTunedModelSpec(
        key="flan_t5_r64",
        label="FLAN-T5 r64",
        adapter_path=FINE_TUNED_MODELS_DIR / "flan_t5" / "flan-t5_adapters_r64",
        base_model_name="google/flan-t5-base",
        architecture="seq2seq",
        comparison_group="rank",
        batch_size=8,
    ),
    FineTunedModelSpec(
        key="flan_t5_r128",
        label="FLAN-T5 r128",
        adapter_path=FINE_TUNED_MODELS_DIR / "flan_t5" / "flan-t5_adapters_r128",
        base_model_name="google/flan-t5-base",
        architecture="seq2seq",
        comparison_group="rank",
        batch_size=8,
    ),
    FineTunedModelSpec(
        key="flan_t5_qkvo",
        label="FLAN-T5 qkvo",
        adapter_path=FINE_TUNED_MODELS_DIR / "flan_t5" / "flan-t5_adapters_different_modules" / "flan-t5-adapters-qkvo",
        base_model_name="google/flan-t5-base",
        architecture="seq2seq",
        comparison_group="modules",
        batch_size=8,
    ),
    FineTunedModelSpec(
        key="flan_t5_qkvoww",
        label="FLAN-T5 qkvoww",
        adapter_path=FINE_TUNED_MODELS_DIR / "flan_t5" / "flan-t5_adapters_different_modules" / "flan-t5-adapters-qkvoww",
        base_model_name="google/flan-t5-base",
        architecture="seq2seq",
        comparison_group="modules",
        batch_size=8,
    ),
    FineTunedModelSpec(
        key="flan_t5_qvww",
        label="FLAN-T5 qvww",
        adapter_path=FINE_TUNED_MODELS_DIR / "flan_t5" / "flan-t5_adapters_different_modules" / "flan-t5-adapters-qvww",
        base_model_name="google/flan-t5-base",
        architecture="seq2seq",
        comparison_group="modules",
        batch_size=8,
    ),
    FineTunedModelSpec(
        key="flan_t5_ww",
        label="FLAN-T5 ww",
        adapter_path=FINE_TUNED_MODELS_DIR / "flan_t5" / "flan-t5_adapters_different_modules" / "flan-t5-adapters-ww",
        base_model_name="google/flan-t5-base",
        architecture="seq2seq",
        comparison_group="modules",
        batch_size=8,
    ),
]

for spec in MODEL_SPECS:
    if not (spec.adapter_path / "adapter_config.json").exists():
        raise FileNotFoundError(f"Missing adapter for {spec.label}: {spec.adapter_path}")

print("Fine-tuned models to evaluate:")
for spec in MODEL_SPECS:
    print(f"- {spec.label}: {spec.adapter_path.relative_to(PROJECT_ROOT)}")


In [ ]:
# Load dataset and sample evaluation subset

try:
    full_dataset_df = pd.read_json(DATASET_PATH, lines=True)
except ValueError:
    full_dataset_df = pd.read_json(DATASET_PATH)

required_columns = {"headline", "is_sarcastic", "reversed_headline"}
missing_columns = required_columns.difference(full_dataset_df.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

full_dataset_df = full_dataset_df.dropna(subset=["headline", "is_sarcastic", "reversed_headline"]).copy()
full_dataset_df["headline"] = full_dataset_df["headline"].astype(str).str.strip()
full_dataset_df["reversed_headline"] = full_dataset_df["reversed_headline"].astype(str).str.strip()
full_dataset_df["is_sarcastic"] = full_dataset_df["is_sarcastic"].astype(int)
full_dataset_df = full_dataset_df[(full_dataset_df["headline"] != "") & (full_dataset_df["reversed_headline"] != "")].copy()

invalid_labels = set(full_dataset_df["is_sarcastic"].unique()) - {0, 1}
if invalid_labels:
    raise ValueError(f"is_sarcastic must contain only 0/1 values. Found: {sorted(invalid_labels)}")

full_dataset_df["source_style"] = full_dataset_df["is_sarcastic"].map(get_source_style)
full_dataset_df["target_style"] = full_dataset_df["is_sarcastic"].map(get_target_style)
full_dataset_df["target_publication"] = full_dataset_df["is_sarcastic"].map(get_target_publication)

dataset_df = sample_dataset(
    dataset_df=full_dataset_df.reset_index(drop=True),
    sample_fraction=DATASET_SAMPLE_PERCENT / 100.0,
    seed=SEED,
)

display(dataset_df[["headline", "is_sarcastic", "reversed_headline"]].head())
print(f"Full dataset examples: {len(full_dataset_df):,}")
print(f"Evaluation examples after sampling: {len(dataset_df):,}")
print("Label counts in sampled evaluation set:")
print(dataset_df["is_sarcastic"].value_counts().sort_index())


In [ ]:
# Adapter inference and scoring helpers

def load_adapter_for_inference(spec: FineTunedModelSpec):
    model_kwargs = {}
    if DEVICE == "cuda" and USE_FP16_ON_GPU:
        model_kwargs["torch_dtype"] = torch.float16

    try:
        tokenizer = AutoTokenizer.from_pretrained(spec.adapter_path)
    except Exception as exc:
        print(f"Could not load tokenizer from {spec.adapter_path}. Falling back to {spec.base_model_name}. Reason: {exc}")
        tokenizer = AutoTokenizer.from_pretrained(spec.base_model_name)
    if spec.architecture == "causal":
        tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_cls = AutoModelForSeq2SeqLM if spec.architecture == "seq2seq" else AutoModelForCausalLM
    base_model = model_cls.from_pretrained(spec.base_model_name, **model_kwargs)
    if tokenizer.pad_token_id is not None:
        base_model.config.pad_token_id = tokenizer.pad_token_id
        if getattr(base_model.generation_config, "pad_token_id", None) is None:
            base_model.generation_config.pad_token_id = tokenizer.pad_token_id

    model = PeftModel.from_pretrained(base_model, spec.adapter_path)
    model.to(DEVICE)
    model.eval()
    return tokenizer, model


def generate_for_adapter(spec: FineTunedModelSpec, eval_df: pd.DataFrame) -> pd.DataFrame:
    generation_path = OUTPUT_DIR / f"{OUTPUT_STEM}_{spec.key}_generations.csv"
    tokenizer, model = load_adapter_for_inference(spec)

    generation_kwargs = {
        "max_new_tokens": spec.max_new_tokens,
        "do_sample": False,
        "num_beams": 1,
        "min_new_tokens": 4,
        "repetition_penalty": 1.05,
    }

    generated_headlines = []
    try:
        for start_idx in tqdm(range(0, len(eval_df), spec.batch_size), desc=f"Generating {spec.label}"):
            batch_df = eval_df.iloc[start_idx:start_idx + spec.batch_size]
            prompts = [
                build_finetuned_prompt(headline=headline, label=label, architecture=spec.architecture)
                for headline, label in zip(batch_df["headline"].tolist(), batch_df["is_sarcastic"].tolist())
            ]
            encoded = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=spec.max_source_length,
            )
            encoded = {key: value.to(DEVICE) for key, value in encoded.items()}

            with torch.inference_mode():
                generated_ids = model.generate(
                    **encoded,
                    pad_token_id=tokenizer.pad_token_id,
                    **generation_kwargs,
                )

            if spec.architecture == "causal":
                input_length = encoded["input_ids"].shape[1]
                decoded = [
                    tokenizer.decode(sequence_ids[input_length:], skip_special_tokens=True)
                    for sequence_ids in generated_ids.detach().cpu()
                ]
            else:
                decoded = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

            generated_headlines.extend(clean_generation(text) for text in decoded)
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    result_df = eval_df[["headline", "is_sarcastic", "source_style", "target_style", "target_publication"]].copy()
    result_df["model_key"] = spec.key
    result_df["model_label"] = spec.label
    result_df["model_name"] = str(spec.adapter_path.relative_to(PROJECT_ROOT))
    result_df["model_architecture"] = spec.architecture
    result_df["comparison_group"] = spec.comparison_group
    result_df["generated_headline"] = generated_headlines
    result_df.to_csv(generation_path, index=False)
    print(f"Saved generations to: {generation_path}")
    return result_df


def build_gpt4o_results(eval_df: pd.DataFrame) -> pd.DataFrame:
    result_df = eval_df[["headline", "is_sarcastic", "source_style", "target_style", "target_publication"]].copy()
    result_df["model_key"] = "gpt4o"
    result_df["model_label"] = "GPT-4o"
    result_df["model_name"] = "dataset.reversed_headline"
    result_df["model_architecture"] = "external"
    result_df["comparison_group"] = "rank"
    result_df["generated_headline"] = eval_df["reversed_headline"].astype(str).tolist()
    return result_df


def compute_cosine_in_batches(source_texts: list[str], generated_texts: list[str]) -> list[float]:
    scores = []
    for start_idx in tqdm(range(0, len(source_texts), SIMILARITY_BATCH_SIZE), desc="Cosine similarity", leave=False):
        end_idx = start_idx + SIMILARITY_BATCH_SIZE
        scores.extend(batch_cosine_similarity(source_texts[start_idx:end_idx], generated_texts[start_idx:end_idx]))
    return scores


def score_results(results_df: pd.DataFrame) -> pd.DataFrame:
    scored_df = results_df.copy()
    scored_df["generated_headline"] = scored_df["generated_headline"].fillna("").astype(str).str.strip()
    valid_mask = scored_df["generated_headline"].ne("")

    scored_df["expected_rewrite_label"] = 1 - scored_df["is_sarcastic"].astype(int)
    scored_df["cosine_similarity"] = math.nan
    scored_df["perplexity"] = math.nan
    scored_df["classifier_predicted_label"] = math.nan
    scored_df["classifier_confidence"] = math.nan
    scored_df["classifier_sarcastic_probability"] = math.nan
    scored_df["classifier_correct"] = False
    scored_df["empty_output"] = ~valid_mask
    scored_df["rewrite_changed"] = (
        scored_df["headline"].str.strip().str.lower()
        != scored_df["generated_headline"].str.strip().str.lower()
    )

    if valid_mask.any():
        valid_df = scored_df.loc[valid_mask].copy()
        valid_df["cosine_similarity"] = compute_cosine_in_batches(
            valid_df["headline"].tolist(),
            valid_df["generated_headline"].tolist(),
        )
        valid_df["perplexity"] = batch_perplexity(
            valid_df["generated_headline"].tolist(),
            device=DEVICE,
            batch_size=PERPLEXITY_BATCH_SIZE,
        )
        predictions, confidences, sarcastic_probabilities = predict_sarcasm_labels(
            texts=valid_df["generated_headline"].tolist(),
            tokenizer=classifier_tokenizer,
            model=classifier_model,
            device=CLASSIFIER_DEVICE,
            batch_size=CLASSIFIER_BATCH_SIZE,
            max_length=CLASSIFIER_MAX_LENGTH,
        )
        valid_df["classifier_predicted_label"] = predictions
        valid_df["classifier_confidence"] = confidences
        valid_df["classifier_sarcastic_probability"] = sarcastic_probabilities
        valid_df["classifier_correct"] = (
            valid_df["classifier_predicted_label"].astype(int) == valid_df["expected_rewrite_label"].astype(int)
        )

        update_columns = [
            "cosine_similarity",
            "perplexity",
            "classifier_predicted_label",
            "classifier_confidence",
            "classifier_sarcastic_probability",
            "classifier_correct",
        ]
        scored_df.loc[valid_mask, update_columns] = valid_df[update_columns].to_numpy()

    return scored_df


def summarise_results(results_df: pd.DataFrame) -> pd.DataFrame:
    return (
        results_df.groupby(["model_key", "model_label", "model_name", "model_architecture", "comparison_group"], as_index=False)
        .agg(
            num_examples=("headline", "size"),
            non_empty_rate=("empty_output", lambda s: 1.0 - float(s.mean())),
            changed_rate=("rewrite_changed", "mean"),
            classifier_accuracy=("classifier_correct", "mean"),
            mean_cosine_similarity=("cosine_similarity", "mean"),
            median_cosine_similarity=("cosine_similarity", "median"),
            mean_perplexity=("perplexity", "mean"),
            median_perplexity=("perplexity", "median"),
        )
        .reset_index(drop=True)
    )


def plot_comparison(summary_df: pd.DataFrame, model_order: list[str], title: str) -> None:
    plot_df = summary_df[summary_df["model_label"].isin(model_order)].copy()
    plot_df["model_label"] = pd.Categorical(plot_df["model_label"], categories=model_order, ordered=True)
    plot_df = plot_df.sort_values("model_label")

    fig, axes = plt.subplots(1, 3, figsize=(24, 6))
    fig.suptitle(title, fontsize=18)

    sns.barplot(data=plot_df, x="model_label", y="classifier_accuracy", hue="model_label", ax=axes[0], palette="mako")
    axes[0].set_title("Classifier Accuracy")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_ylim(0, 1)

    sns.barplot(data=plot_df, x="model_label", y="mean_cosine_similarity", hue="model_label", ax=axes[1], palette="crest")
    axes[1].set_title("Meaning Preservation")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("Mean cosine similarity")

    sns.barplot(data=plot_df, x="model_label", y="mean_perplexity", hue="model_label", ax=axes[2], palette="flare")
    axes[2].set_title("Fluency")
    axes[2].set_xlabel("")
    axes[2].set_ylabel("Mean perplexity (lower is better)")

    for ax in axes:
        if ax.legend_ is not None:
            ax.legend_.remove()
        ax.tick_params(axis="x", rotation=25)

    plt.tight_layout()
    plt.show()


In [ ]:
# Run GPT-4o baseline scoring input and fine-tuned adapter inference

all_generation_results = [build_gpt4o_results(dataset_df)]

for spec in MODEL_SPECS:
    print(f"\n=== Running inference for {spec.label} ===")
    generated_df = generate_for_adapter(spec, dataset_df)
    all_generation_results.append(generated_df)

generation_results_df = pd.concat(all_generation_results, ignore_index=True)
generation_results_path = OUTPUT_DIR / f"{OUTPUT_STEM}_fine_tuned_generations_all_models.csv"
generation_results_df.to_csv(generation_results_path, index=False)

print(f"Saved all generations to: {generation_results_path}")
display(generation_results_df.head())


In [ ]:
# Score all generations with classifier accuracy, cosine similarity, and perplexity

classifier_tokenizer, classifier_model = load_sarcasm_classifier(
    model_name=CLASSIFIER_MODEL_NAME,
    device=CLASSIFIER_DEVICE,
)

scored_results = []
for model_label, model_df in generation_results_df.groupby("model_label", sort=False):
    print(f"\n=== Scoring {model_label} ===")
    scored_results.append(score_results(model_df))

benchmark_results_df = pd.concat(scored_results, ignore_index=True)
benchmark_results_path = OUTPUT_DIR / f"{OUTPUT_STEM}_fine_tuned_metrics_all_models.csv"
benchmark_results_df.to_csv(benchmark_results_path, index=False)

summary_df = summarise_results(benchmark_results_df)
summary_path = OUTPUT_DIR / f"{OUTPUT_STEM}_fine_tuned_metrics_summary.csv"
summary_df.to_csv(summary_path, index=False)

print(f"Saved metrics to: {benchmark_results_path}")
print(f"Saved summary to: {summary_path}")
display(summary_df)


In [ ]:
# Comparison graph: GPT-2 vs GPT-4o vs FLAN-T5 rank variants

rank_model_order = [
    "GPT-2 fine-tuned",
    "GPT-4o",
    "FLAN-T5 r32",
    "FLAN-T5 r64",
    "FLAN-T5 r128",
]

plot_comparison(
    summary_df=summary_df,
    model_order=rank_model_order,
    title="GPT-2 vs GPT-4o vs FLAN-T5 LoRA Rank Variants",
)


In [ ]:
# Comparison graph: FLAN-T5 target-module variants

module_model_order = [
    "FLAN-T5 qkvo",
    "FLAN-T5 qkvoww",
    "FLAN-T5 qvww",
    "FLAN-T5 ww",
]

plot_comparison(
    summary_df=summary_df,
    model_order=module_model_order,
    title="FLAN-T5 Target-Module Variants",
)


In [ ]:
# Distribution graph: GPT-2 vs GPT-4o vs FLAN-T5 rank variants

def plot_metric_distributions(results_df: pd.DataFrame, model_order: list[str], title: str) -> None:
    plot_df = results_df[
        results_df["model_label"].isin(model_order)
        & results_df["generated_headline"].fillna("").str.strip().ne("")
    ].copy()
    if plot_df.empty:
        raise ValueError(f"No non-empty generated outputs available for: {model_order}")

    plot_df["model_label"] = pd.Categorical(plot_df["model_label"], categories=model_order, ordered=True)
    plot_df = plot_df.sort_values("model_label")

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(title, fontsize=18)

    sns.boxplot(
        data=plot_df,
        x="model_label",
        y="cosine_similarity",
        hue="model_label",
        ax=axes[0],
        palette="crest",
    )
    axes[0].set_title("Cosine Similarity Distribution")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Cosine similarity")

    sns.boxplot(
        data=plot_df,
        x="model_label",
        y="perplexity",
        hue="model_label",
        ax=axes[1],
        palette="flare",
        showfliers=False,
    )
    axes[1].set_title("Perplexity Distribution")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("Perplexity")

    for ax in axes:
        if ax.legend_ is not None:
            ax.legend_.remove()
        ax.tick_params(axis="x", rotation=25)

    plt.tight_layout()
    plt.show()


plot_metric_distributions(
    results_df=benchmark_results_df,
    model_order=rank_model_order,
    title="Metric Distributions: GPT-2 vs GPT-4o vs FLAN-T5 Rank Variants",
)


In [ ]:
# Distribution graph: FLAN-T5 target-module variants

plot_metric_distributions(
    results_df=benchmark_results_df,
    model_order=module_model_order,
    title="Metric Distributions: FLAN-T5 Target-Module Variants",
)


In [ ]:
# Qualitative preview

preview_rows = []
for model_label, frame in benchmark_results_df.groupby("model_label", sort=False):
    preview_rows.append(frame.sample(n=min(3, len(frame)), random_state=SEED))

preview_df = pd.concat(preview_rows, ignore_index=True)[[
    "model_label",
    "source_style",
    "target_style",
    "headline",
    "generated_headline",
    "classifier_correct",
    "cosine_similarity",
    "perplexity",
]]

display(preview_df)
